# 02. ROS 2 rclpy — Google Colab 에서 Publisher / Subscriber

Colab 은 **Ubuntu 22.04 x86_64** 리눅스 VM 이라 ROS 2 를 apt 로 설치할 수 있습니다. Ubuntu 22.04 의 짝은 **Humble** 이므로, 이 노트북은 ROS 2 **Humble** 을 씁니다 (개념은 Jazzy 와 100% 동일).

![ROS](https://upload.wikimedia.org/wikipedia/commons/b/bb/Ros_logo.svg)

### 배울 것
1. Colab 에 ROS 2 Humble 설치 (1 회, 약 2~3분)
2. Python publisher / subscriber 스크립트 작성
3. 백그라운드 subprocess 로 두 노드 동시 실행
4. 메시지 캡처 → matplotlib 로 시각화
5. `ros2 topic list`, `ros2 node list` 셸 명령 실습

> 💡 Colab 에는 GUI 가 없으므로 turtlesim 은 대신 Mac 로컬 Docker 에서 돌리세요. 이 노트북은 **순수 Python + 토픽** 만 다룹니다.

## 1. ROS 2 Humble 설치

Colab 런타임을 새로 시작할 때마다 1회 실행. 약 2~3분 소요.

In [ ]:
%%bash
# Ubuntu 22.04 확인
lsb_release -a 2>/dev/null | head -3
echo '---'
# ROS 2 Humble apt 소스 등록
sudo apt update -qq
sudo apt install -y -qq software-properties-common curl gnupg lsb-release locales > /dev/null
sudo locale-gen en_US en_US.UTF-8 > /dev/null
sudo update-locale LC_ALL=en_US.UTF-8 LANG=en_US.UTF-8 > /dev/null
sudo add-apt-repository -y universe > /dev/null 2>&1

sudo curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu $(. /etc/os-release && echo $UBUNTU_CODENAME) main" | sudo tee /etc/apt/sources.list.d/ros2.list > /dev/null

sudo apt update -qq
sudo apt install -y -qq ros-humble-ros-base python3-colcon-common-extensions > /dev/null
echo 'ROS 2 Humble 설치 완료'

In [ ]:
%%bash
source /opt/ros/humble/setup.bash
echo "ROS_DISTRO=$ROS_DISTRO"
ros2 --help | head -15

## 2. Publisher 스크립트 작성

Colab 의 `%%writefile` 매직으로 Python 파일을 디스크에 저장.

In [ ]:
%%writefile publisher.py
import rclpy
from rclpy.node import Node
from std_msgs.msg import Float64
import math, time


class SinePub(Node):
    def __init__(self):
        super().__init__('sine_pub')
        self.pub = self.create_publisher(Float64, 'sine', 10)
        self.t0 = time.time()
        self.create_timer(0.05, self.cb)   # 20 Hz

    def cb(self):
        t = time.time() - self.t0
        msg = Float64()
        msg.data = math.sin(2 * math.pi * 0.5 * t)   # 0.5 Hz 사인파
        self.pub.publish(msg)


def main():
    rclpy.init()
    rclpy.spin(SinePub())


if __name__ == '__main__':
    main()


## 3. Subscriber 스크립트 작성

받은 메시지를 CSV 로 기록해서 나중에 시각화합니다.

In [ ]:
%%writefile subscriber.py
import rclpy, time, csv, sys
from rclpy.node import Node
from std_msgs.msg import Float64


class SineSub(Node):
    def __init__(self):
        super().__init__('sine_sub')
        self.sub = self.create_subscription(Float64, 'sine', self.cb, 10)
        self.t0 = time.time()
        self.fp = open('sine_log.csv', 'w', buffering=1)
        self.w = csv.writer(self.fp)
        self.w.writerow(['t', 'value'])

    def cb(self, msg):
        self.w.writerow([f'{time.time()-self.t0:.3f}', msg.data])


def main():
    rclpy.init()
    node = SineSub()
    try:
        rclpy.spin(node)
    finally:
        node.fp.close()


if __name__ == '__main__':
    main()


## 4. 두 노드 백그라운드 실행

Colab 셸에서 `ros2 run` 대신 바로 `python` 으로 실행 (더 빠르고 ament 빌드 불필요).
- `nohup ... &` 로 백그라운드
- 6초 후 강제 종료
- `sine_log.csv` 에 쌓인 데이터를 시각화

In [ ]:
import subprocess, os, time, signal

os.environ['ROS_DOMAIN_ID'] = '42'

def spawn(cmd):
    return subprocess.Popen(
        ['bash', '-lc', 'source /opt/ros/humble/setup.bash && ' + cmd],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        preexec_fn=os.setsid,
    )

pub = spawn('python publisher.py')
sub = spawn('python subscriber.py')

try:
    time.sleep(6.0)
finally:
    for proc in (pub, sub):
        os.killpg(os.getpgid(proc.pid), signal.SIGINT)
    time.sleep(0.5)
print('노드 2개 실행 완료 → sine_log.csv 확인')

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv('sine_log.csv')
print(f'received {len(df)} messages')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(df['t'], df['value'], lw=1.5)
ax.set_xlabel('time [s]')
ax.set_ylabel('value')
ax.set_title('Topic /sine — received by subscriber')
ax.grid(alpha=0.3)
ax.axhline(0, color='gray', lw=0.5)
plt.show()

## 5. `ros2` CLI 도구도 Colab 에서 쓸 수 있다

발행 중 (실행이 빠르게 지나가니 두 터미널 없이 직접 체험하긴 어렵지만) 아래처럼 CLI 사용법은 동일:

In [ ]:
%%bash
source /opt/ros/humble/setup.bash
# publisher 다시 잠깐 띄움
python publisher.py &
PUB_PID=$!
sleep 1.5
echo '=== topic list ==='
ros2 topic list
echo
echo '=== topic info /sine ==='
ros2 topic info /sine
echo
echo '=== first 3 messages ==='
timeout 1.5 ros2 topic echo /sine --once
kill $PUB_PID 2>/dev/null
wait $PUB_PID 2>/dev/null

## 6. Mac 쪽 Docker 컨테이너와 연결하고 싶다면?

Colab 은 Google 클라우드 VM 이므로 NAT 뒤에 있어 내 Mac 의 Docker 와 직접 DDS 통신은 **어렵습니다**. ROS 2 통신은 기본적으로 같은 L2/L3 네트워크 세그먼트를 전제로 하기 때문입니다.

대안:
- **`rosbridge_suite`** (websocket) 를 Docker 쪽에 띄우고 Colab 의 Python 이 `roslibpy` 로 연결 → 공인 IP/포트 포워딩 필요
- **`Zenoh`** 기반 DDS 브리지 → 하지만 여전히 NAT 뚫기 어렵습니다
- 학습용으로는 **같은 호스트(Mac 로컬 Docker 안 ros2 / Mac venv rclpy)** 또는 **Colab 내부에서 완결** 하는 게 쉽습니다.

## 7. 정리

- Colab 에서 ROS 2 를 apt 로 설치 → **몇 줄 Python** 으로 pub/sub 확인 가능
- `subprocess.Popen` 으로 여러 노드 병렬 실행
- 토픽 데이터는 CSV → pandas → matplotlib 로 시각화
- 이 구조를 그대로 다음 노트북에서 **PyBullet 시뮬과 엮어** 로봇 상태를 토픽으로 흘려봅니다.